# Tutorial 8.2: MPC under uncertainty and controller evaluation

This tutorial evaluates the same MPC formulation under forecast, model and measurement errors. All scenarios use the same reporting period and metric definitions. We also compare MPC with a fixed open-loop schedule to show the value of feedback under uncertainty.

## Learning outcomes

By the end of the tutorial, you should be able to:

1. distinguish forecast, model and measurement errors;
2. calculate ex-post operational metrics from the reference plant;
3. compare fixed and feedback controllers under a common realised scenario; and
4. explain the benefits and limits of feedback and constraint tightening.

## Indicative schedule

- experiment design and metrics: 20 minutes;
- scenario runs: 30 minutes;
- fixed versus feedback control: 20 minutes;
- robustness and sensitivity plots: 15 minutes;
- interpretation: 10 minutes;
- discussion: 5 minutes.

## 1. Set-up and common evaluation period

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from course_utils.mpc import run_fixed_schedule_experiment, run_mpc_experiment

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    raise FileNotFoundError("Run this notebook from the repository root.")

plt.rcParams.update({"figure.figsize": (10, 4), "figure.dpi": 110})

In [ ]:
operation = pd.read_csv(
    ROOT / "data" / "reference_building" / "operation_conditions.csv",
    index_col="timestamp",
    parse_dates=True,
).sort_index()
forecasts = pd.read_csv(
    ROOT / "data" / "reference_building" / "operation_forecasts.csv",
    parse_dates=["issue_time", "valid_time"],
)
model = json.loads(
    (ROOT / "data" / "reference_building" / "verified_controller_model.json").read_text()
)
evaluation_conditions = operation.iloc[:48]

## 2. Task 1 — realised metrics

Ex-ante quantities come from the controller model before an action is applied. Ex-post metrics use realised reference-plant temperature and power. Controller claims should use the latter.

In [ ]:
def controller_metrics(results, conditions):
    power = results["heat_pump_electric_kw"].to_numpy()
    temperature = results["next_indoor_temperature_c"].to_numpy()
    lower = conditions["comfort_min_c"].to_numpy()
    violation = np.maximum(lower - temperature, 0.0)
    dt = 0.5
    return {
        "Energy [kWh]": float(power.sum() * dt),
        "Cost [£]": float(np.sum(power * conditions["import_price_gbp_per_kwh"].to_numpy()) * dt),
        "Peak [kW]": float(power.max()),
        "Violation [h]": float(np.sum(violation > 0) * dt),
        "Degree-hours [°C h]": float(violation.sum() * dt),
        "Maximum deviation [°C]": float(violation.max()),
        "Mean solve time [s]": float(results["solve_time_s"].mean()),
        "Fallback count": int(results["fallback"].sum()),
    }

## 3. Task 2 — define uncertainty scenarios

Use one nominal case, a cold-biased forecast, a plant with higher heat loss, a plant with higher thermal mass, a warm sensor bias and a tightened comfort band. Each scenario changes one assumption while holding the reporting period and controller structure fixed.

In [ ]:
scenarios = {
    "Nominal": {},
    "Cold-biased forecast": {"forecast_temperature_bias_c": -1.5},
    "Higher heat loss": {"plant_resistance_multiplier": 0.80},
    "Higher thermal mass": {"plant_capacitance_multiplier": 1.30},
    "Warm sensor bias": {"sensor_bias_c": 0.40},
    "Tightened comfort band": {"comfort_margin_c": 0.30},
}

In [ ]:
scenario_results = {}
scenario_metrics = {}
for name, arguments in scenarios.items():
    result = run_mpc_experiment(
        operation,
        forecasts,
        model,
        evaluation_steps=48,
        horizon_steps=48,
        random_state=90,
        **arguments,
    )
    scenario_results[name] = result
    scenario_metrics[name] = controller_metrics(result, evaluation_conditions)

metric_table = pd.DataFrame(scenario_metrics).T
display(metric_table.round(3))

## 4. Compare realised trajectories

In [ ]:
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(10, 8))
for name in ["Nominal", "Cold-biased forecast", "Higher heat loss", "Warm sensor bias"]:
    if name in scenario_results:
        axes[0].plot(
            scenario_results[name]["next_indoor_temperature_c"],
            label=name,
        )
axes[0].plot(evaluation_conditions["comfort_min_c"], "--", color="black", label="Comfort bounds")
axes[0].plot(evaluation_conditions["comfort_max_c"], "--", color="black")
axes[0].set_ylabel("Temperature [°C]")
axes[0].legend(ncol=2)
for name in ["Nominal", "Cold-biased forecast", "Higher heat loss", "Warm sensor bias"]:
    if name in scenario_results:
        axes[1].step(
            scenario_results[name].index,
            scenario_results[name]["heat_pump_electric_kw"],
            where="post",
            label=name,
        )
axes[1].set_ylabel("Heat-pump power [kW]")
axes[1].set_xlabel("Time")
axes[1].legend(ncol=2)
plt.show()

## 5. Task 3 — fixed schedule versus MPC under uncertainty

A fixed controller optimises once and cannot react when the realised building departs from its model. MPC receives a new indoor-temperature measurement every 30 minutes and can correct the remaining schedule.

Use a deliberately severe heat-loss error as a stress test: the reference plant's thermal resistance is half the assumed value. Both controllers face the same weather, plant parameters, noise sequence and reporting period. The purpose is to expose the feedback mechanism, not to claim that such a large parameter error is typical.

In [ ]:
stress_scenario = {"plant_resistance_multiplier": 0.50}

fixed_stress_results = run_fixed_schedule_experiment(
    operation,
    forecasts,
    model,
    evaluation_steps=48,
    horizon_steps=48,
    random_state=90,
    **stress_scenario,
)
mpc_stress_results = run_mpc_experiment(
    operation,
    forecasts,
    model,
    evaluation_steps=48,
    horizon_steps=48,
    random_state=90,
    **stress_scenario,
)

feedback_comparison = pd.DataFrame(
    {
        "Fixed open-loop schedule": controller_metrics(
            fixed_stress_results, evaluation_conditions
        ),
        "MPC with feedback": controller_metrics(
            mpc_stress_results, evaluation_conditions
        ),
    }
).T
display(feedback_comparison.round(3))

discomfort_reduction_percent = 100 * (
    1
    - feedback_comparison.loc["MPC with feedback", "Degree-hours [°C h]"]
    / feedback_comparison.loc[
        "Fixed open-loop schedule", "Degree-hours [°C h]"
    ]
)
print(
    "MPC reduction in degree-hours relative to the fixed schedule: "
    f"{discomfort_reduction_percent:.1f}%"
)
assert (
    feedback_comparison.loc["MPC with feedback", "Degree-hours [°C h]"]
    < feedback_comparison.loc[
        "Fixed open-loop schedule", "Degree-hours [°C h]"
    ]
)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 9))
axes[0].plot(
    fixed_stress_results["next_indoor_temperature_c"],
    label="Fixed open-loop schedule",
)
axes[0].plot(
    mpc_stress_results["next_indoor_temperature_c"],
    label="MPC with feedback",
)
axes[0].plot(
    evaluation_conditions["comfort_min_c"],
    "--",
    color="black",
    label="Comfort bounds",
)
axes[0].plot(evaluation_conditions["comfort_max_c"], "--", color="black")
axes[0].set_ylabel("Temperature [°C]")
axes[0].legend()
axes[1].step(
    fixed_stress_results.index,
    fixed_stress_results["heat_pump_electric_kw"],
    where="post",
    label="Fixed open-loop schedule",
)
axes[1].step(
    mpc_stress_results.index,
    mpc_stress_results["heat_pump_electric_kw"],
    where="post",
    label="MPC with feedback",
)
axes[1].set_ylabel("Heat-pump power [kW]")
axes[1].legend()
axes[2].bar(
    feedback_comparison.index,
    feedback_comparison["Degree-hours [°C h]"],
)
axes[2].set_ylabel("Discomfort [°C h]")
axes[2].tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

The feedback controller uses more energy in this stress test, but it substantially reduces accumulated discomfort and maximum temperature deviation. This is the central comparison: feedback does not remove uncertainty, but it prevents state error from remaining uncorrected throughout the full schedule.

## 6. Prediction-horizon sensitivity

A longer horizon can anticipate prices and occupancy earlier, but it relies on more distant forecasts and takes more time to solve.

In [ ]:
horizon_metrics = {}
for horizon_steps in [12, 24, 48]:
    result = run_mpc_experiment(
        operation,
        forecasts,
        model,
        evaluation_steps=48,
        horizon_steps=horizon_steps,
        random_state=90,
    )
    horizon_metrics[f"{horizon_steps / 2:.0f} h"] = controller_metrics(
        result, evaluation_conditions
    )

horizon_table = pd.DataFrame(horizon_metrics).T
display(horizon_table.round(3))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
horizon_table["Cost [£]"].plot.bar(ax=axes[0], title="Realised cost")
horizon_table["Degree-hours [°C h]"].plot.bar(ax=axes[1], title="Realised discomfort")
axes[0].set_ylabel("Cost [£]")
axes[1].set_ylabel("Degree-hours [°C h]")
plt.tight_layout()
plt.show()

## Discussion

1. Which error source has the largest operational effect in this experiment?
2. Does the cold-biased forecast necessarily worsen comfort?
3. Why does the fixed schedule accumulate more discomfort in the severe heat-loss case?
4. What does MPC trade for its improvement in comfort?
5. What does constraint tightening trade against improved robustness?
6. Why can feedback fail to correct an error immediately?
7. Why should controller tuning scenarios remain separate from final evaluation scenarios?